# Lecture 11: Speech Data I

So far in this course, we've been working exclusively with text — written
words in strings, files, and corpora. But language isn't just written. Most
of the world's languages are primarily spoken, and even for well-documented
written languages, speech carries information (intonation, emphasis, speaker
identity) that text simply doesn't capture.

Today we'll start working with speech data:
- What is sound, physically?
- How do we capture and represent it digitally?
- What can we see when we visualize speech?

**Prerequisites:**
- Python and the `ling250` environment
- Familiarity with numpy arrays and matplotlib from earlier lectures

---

## Part 1: What is Sound?

Sound is a **pressure wave** — a vibration that travels through air (or
another medium). When you speak, your vocal folds and articulators create
rapid changes in air pressure. Those pressure changes travel outward from
your mouth and eventually reach a listener's ear (or a microphone).

Three physical properties determine what a sound "sounds like":

| Property | Physical quantity | Perceptual correlate |
|----------|-------------------|---------------------|
| **Amplitude** | Size of pressure changes | Loudness |
| **Frequency** | Rate of vibration (cycles/sec = Hz) | Pitch |
| **Waveform shape** | Pattern of the pressure changes | Timbre (sound quality) |

The simplest possible sound is a **pure tone** — a single-frequency
vibration that produces a smooth sine wave. Real sounds (speech, music,
noise) are complex combinations of many frequencies at once.

A 100 Hz tone means the air pressure completes 100 full cycles of
oscillation every second. A 200 Hz tone vibrates twice as fast and sounds
higher in pitch.

---

## Part 2: Waveforms

A **waveform** is a plot of amplitude over time — it shows the raw pressure
changes that make up a sound. This is the most basic visualization of audio.

We'll use the `librosa` library to load audio files and `matplotlib` to
plot them.

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load a speech recording (a Harvard sentence, recorded at 16 kHz)
# sr=None tells librosa to use the file's native sample rate
speech_signal, speech_sr = librosa.load('../audio/kdt_001.wav', sr=None)

print(f"Sample rate: {speech_sr} Hz")
print(f"Duration: {len(speech_signal) / speech_sr:.2f} seconds")
print(f"Number of samples: {len(speech_signal):,}")

In [ ]:
# Plot the waveform: amplitude on the y-axis, time on the x-axis
plt.figure(figsize=(12, 3))
librosa.display.waveshow(speech_signal, sr=speech_sr)
plt.title('Speech waveform: kdt_001.wav')
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

The speech waveform looks messy and complex — that's normal. Speech is a
rapidly changing mixture of many frequencies. You can see louder parts
(larger amplitude) and quieter parts, and some gaps that likely correspond
to pauses or voiceless sounds.

Now let's compare this to a pure synthetic sine wave:

In [ ]:
# Load a synthetic 100 Hz sine wave
sine_signal, sine_sr = librosa.load('../audio/sine_100Hz.wav', sr=None)

# How many samples correspond to 100 ms (10 cycles at 100 Hz)
zoom_samples = int(0.1 * sine_sr)
zoom_time_ms = np.arange(zoom_samples) / sine_sr * 1000

fig, axes = plt.subplots(2, 1, figsize=(12, 5))

# Top: full duration — looks like a solid rectangle at this scale
librosa.display.waveshow(sine_signal, sr=sine_sr, ax=axes[0])
axes[0].set_title('Sine wave: 100 Hz (full duration)')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Amplitude')

# Bottom: first 100 ms — 10 complete cycles, shape is visible
axes[1].plot(zoom_time_ms, sine_signal[:zoom_samples])
axes[1].set_title('Zoomed in: first 100 ms (10 cycles)')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

At full duration (top), the sine wave looks like a solid rectangle — the
100 cycles per second are too dense to see individually at this scale.
Zoom in to just 100 ms (bottom) and the smooth, regularly repeating shape
becomes clear. This is a pure tone: the same waveform, over and over,
with no variation.

Real speech (above) never looks like this — it's constantly changing as
your articulators move and different sounds are produced.

Let's zoom into a small window of the speech signal to see individual wave
cycles there too:

In [ ]:
# Zoom into 30 milliseconds of speech (starting at 0.5 seconds)
start_sample = int(0.5 * speech_sr)
window_samples = int(0.03 * speech_sr)
end_sample = start_sample + window_samples

# Create a time axis in milliseconds
time_ms = np.arange(window_samples) / speech_sr * 1000

plt.figure(figsize=(12, 3))
plt.plot(time_ms, speech_signal[start_sample:end_sample])
plt.title('Zoomed speech waveform (30 ms window)')
plt.xlabel('Time (ms)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

At this zoom level, you can start to see a repeating pattern — that's the
vibration of the speaker's vocal folds. The repetition rate is the
**fundamental frequency (F0)**, which we'll come back to later.

---

## Part 3: Digital Sampling

Sound in the real world is a **continuous** signal — air pressure varies
smoothly over time. But computers work with **discrete** numbers. To store
sound digitally, we need to convert the continuous signal into a sequence of
numbers. This process is called **analog-to-digital conversion (ADC)**,
and it involves two key choices:

**Sample rate** — how many measurements we take per second, measured in Hz.
A sample rate of 16,000 Hz means we record 16,000 amplitude measurements
every second.

**Bit depth** — how precisely we measure each sample. With 16-bit audio,
each sample can take one of 2^16 = 65,536 possible values. Higher bit
depth means finer distinctions in amplitude (less "rounding").

| Sample rate | Context | Frequency range |
|-------------|---------|----------------|
| 8,000 Hz | Telephone | Up to 4 kHz |
| 16,000 Hz | Speech research | Up to 8 kHz |
| 22,050 Hz | Low-quality audio | Up to 11 kHz |
| 44,100 Hz | CD quality | Up to 22 kHz |
| 48,000 Hz | Professional audio | Up to 24 kHz |

For speech research, 16 kHz is the standard — it captures all the
frequencies relevant to understanding speech without wasting storage on
ultrasonic frequencies that don't matter.

In [ ]:
# Zoom in far enough to see individual samples as discrete points
# Show 5 ms of the sine wave — at 16 kHz that's 80 samples
n_samples = int(0.005 * sine_sr)
time_ms = np.arange(n_samples) / sine_sr * 1000

fig, axes = plt.subplots(1, 2, figsize=(14, 3))

# Left: connected line (looks continuous)
axes[0].plot(time_ms, sine_signal[:n_samples])
axes[0].set_title('Sine wave (line plot)')
axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('Amplitude')

# Right: individual samples as dots
axes[1].stem(time_ms, sine_signal[:n_samples], linefmt='C0-',
             markerfmt='C0o', basefmt='k-')
axes[1].set_title('Same data — individual samples')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

The left plot *looks* continuous, but that's just matplotlib connecting the
dots. The right plot shows what we actually have: a finite set of discrete
amplitude measurements. Digital audio is always a sequence of numbers —
the illusion of continuity comes from having enough samples per second.

---

## Part 4: Nyquist Frequency and Aliasing

There's a fundamental limit on what frequencies a digital recording can
capture. To faithfully represent a frequency, you need **at least 2 samples
per cycle**. This gives us:

> **Nyquist frequency** = sample rate / 2

This is the highest frequency that can be accurately represented at a
given sample rate. For our 16 kHz speech recordings:

- Nyquist frequency = 16,000 / 2 = **8,000 Hz**
- Any frequency component above 8,000 Hz cannot be captured

This is fine for speech — the fundamental frequency of human speech
typically ranges from about 85 Hz (low male voice) to 300 Hz (high
female/child voice), and even the highest-frequency speech sounds
(like /s/) have their energy concentrated below 8 kHz.

**Aliasing** is what happens when you try to record a frequency above the
Nyquist limit — the signal "folds back" and appears as a lower frequency
that wasn't actually in the original sound. Recording equipment uses
**anti-aliasing filters** to remove frequencies above the Nyquist limit
before sampling.

---

## Part 5: Audio Formats

Once audio has been digitized, it needs to be stored in a file. Audio
formats fall into two categories:

| Format | Type | Compression | Notes |
|--------|------|-------------|-------|
| **WAV** | Lossless | None | Raw samples; large files. The standard for research. |
| **FLAC** | Lossless | Yes | Compressed but perfectly reconstructible. ~60% of WAV size. |
| **MP3** | Lossy | Yes | Discards "inaudible" information. ~10% of WAV size. |
| **OGG** | Lossy | Yes | Open-source alternative to MP3. Similar quality/size. |

**Lossless** formats preserve every single sample — you can convert
WAV → FLAC → WAV and get back exactly what you started with. FLAC
achieves smaller files by compressing the data (like a zip file), not by
throwing anything away.

**Lossy** formats achieve much smaller files by removing information that
psychoacoustic models predict humans won't notice. This is fine for
listening, but those removed details might matter for acoustic analysis.

**Rule of thumb for research:** Always work with lossless formats (WAV or
FLAC). Use lossy formats only for distribution (e.g., sharing audio
examples on a website). Once information is lost to lossy compression,
it cannot be recovered.

---

## Part 6: The Fourier Transform

Here's a key insight: **any complex periodic sound can be decomposed into
a sum of sine waves** at different frequencies and amplitudes. This is
the idea behind the **Fourier Transform** — one of the most important
tools in signal processing.

Let's build intuition by going in the forward direction first: combining
simple sine waves into a complex signal.

In [ ]:
# Generate individual sine waves at different frequencies
duration = 0.05  # 50 ms — enough to see the pattern
sample_rate = 16000
t = np.arange(int(duration * sample_rate)) / sample_rate

sine_100 = np.sin(2 * np.pi * 100 * t)
sine_200 = 0.7 * np.sin(2 * np.pi * 200 * t)
sine_300 = 0.5 * np.sin(2 * np.pi * 300 * t)

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)

time_ms = t * 1000
axes[0].plot(time_ms, sine_100)
axes[0].set_title('100 Hz (amplitude = 1.0)')
axes[0].set_ylabel('Amplitude')

axes[1].plot(time_ms, sine_200)
axes[1].set_title('200 Hz (amplitude = 0.7)')
axes[1].set_ylabel('Amplitude')

axes[2].plot(time_ms, sine_300)
axes[2].set_title('300 Hz (amplitude = 0.5)')
axes[2].set_ylabel('Amplitude')

# Sum them together
composite = sine_100 + sine_200 + sine_300
axes[3].plot(time_ms, composite)
axes[3].set_title('Composite: 100 + 200 + 300 Hz')
axes[3].set_xlabel('Time (ms)')
axes[3].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

The composite waveform at the bottom looks more complex than any of the
individual components — but it's still just three sine waves added
together. Real speech works the same way, just with many more components.

The **Fourier Transform** does the reverse: given a complex waveform, it
tells you which frequencies are present and how strong each one is.

- **Input:** a waveform (amplitude over time)
- **Output:** a **spectrum** (amplitude at each frequency)

The **Fast Fourier Transform (FFT)** is an efficient algorithm for
computing this. We don't need to understand how the algorithm works — just
what goes in and what comes out.

In [ ]:
# Generate a longer composite signal for cleaner FFT results
duration_fft = 0.5  # 500 ms
t_fft = np.arange(int(duration_fft * sample_rate)) / sample_rate

composite_long = (
    1.0 * np.sin(2 * np.pi * 100 * t_fft)
    + 0.7 * np.sin(2 * np.pi * 200 * t_fft)
    + 0.5 * np.sin(2 * np.pi * 300 * t_fft)
)

# Compute the FFT
fft_result = np.fft.fft(composite_long)
fft_freqs = np.fft.fftfreq(len(composite_long), d=1/sample_rate)

# We only need the positive frequencies (the spectrum is symmetric)
positive_mask = fft_freqs >= 0
magnitude = np.abs(fft_result[positive_mask])
freqs = fft_freqs[positive_mask]

plt.figure(figsize=(12, 3))
plt.plot(freqs, magnitude)
plt.xlim(0, 500)
plt.title('Frequency spectrum of composite signal')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude')
plt.tight_layout()
plt.show()

The FFT recovers exactly the three frequencies we put in: peaks at 100, 200,
and 300 Hz, with heights proportional to the amplitudes we used (1.0, 0.7,
0.5). The Fourier Transform has "decomposed" our complex signal back into its
component parts.

Now let's try a real physical example. When a dulcimer string is struck, it
doesn't vibrate at a single frequency — it vibrates at its **fundamental**
(determined by the string's length and tension) *and* at integer multiples of
that fundamental, called **harmonics**. Let's see this in the FFT:

In [ ]:
# Load a dulcimer C note (C4, fundamental ~262 Hz, recorded at 44.1 kHz)
dulcimer_signal, dulcimer_sr = librosa.load('../audio/dulcimer_C.wav', sr=None)

dulcimer_fft = np.fft.fft(dulcimer_signal)
dulcimer_freqs = np.fft.fftfreq(len(dulcimer_signal), d=1/dulcimer_sr)

positive_mask = dulcimer_freqs >= 0
dulcimer_magnitude = np.abs(dulcimer_fft[positive_mask])
dulcimer_freqs_pos = dulcimer_freqs[positive_mask]

# Zoom the waveform to 10 cycles of the fundamental (10 / 262 Hz ≈ 38 ms),
# starting 0.1 seconds in so the initial strike transient has settled
fundamental_hz = 262
offset_samples = int(0.1 * dulcimer_sr)
window_samples = int(10 / fundamental_hz * dulcimer_sr)
zoom_signal = dulcimer_signal[offset_samples : offset_samples + window_samples]
zoom_time_ms = np.arange(window_samples) / dulcimer_sr * 1000

fig, axes = plt.subplots(2, 1, figsize=(12, 5))

# Top: zoomed waveform showing ~10 regular cycles
axes[0].plot(zoom_time_ms, zoom_signal)
axes[0].set_title('Dulcimer C note — waveform (10 cycles, starting at 0.1 s)')
axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('Amplitude')

# Bottom: frequency spectrum
axes[1].plot(dulcimer_freqs_pos, dulcimer_magnitude)
axes[1].set_xlim(0, 3000)
axes[1].set_title('Dulcimer C note — frequency spectrum')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')

plt.tight_layout()
plt.show()

The peaks fall at regular intervals: the **fundamental** (C4 ≈ 262 Hz) and
its harmonics at 2× (~524 Hz), 3× (~786 Hz), 4× (~1048 Hz), and so on. This
evenly-spaced comb of peaks is the signature of any periodically vibrating
object.

This is exactly what vocal fold vibration produces. When you say a vowel,
your vocal folds vibrate at a rate determined by muscle tension — that's the
fundamental frequency (F0) — and generate harmonics at 2×F0, 3×F0, etc. We'll
see this structure in spectrograms in Part 8.

Now let's look at a speech recording:

In [ ]:
# FFT of the speech signal
speech_fft = np.fft.fft(speech_signal)
speech_freqs = np.fft.fftfreq(len(speech_signal), d=1/speech_sr)

positive_mask = speech_freqs >= 0
speech_magnitude = np.abs(speech_fft[positive_mask])
speech_freqs_pos = speech_freqs[positive_mask]

plt.figure(figsize=(12, 3))
plt.plot(speech_freqs_pos, speech_magnitude)
plt.xlim(0, 8000)
plt.title('Frequency spectrum of speech recording')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude')
plt.tight_layout()
plt.show()

The speech spectrum is much more complex — there's energy spread across
many frequencies, with more energy concentrated in the lower frequencies
(below ~4000 Hz). This is typical of speech.

But there's a problem: this spectrum represents the **entire recording**
as a single summary. It tells us what frequencies are present on average,
but speech changes rapidly over time — a vowel has very different frequency
content from a fricative. We need a way to see how the frequency content
changes *over time*. That's what spectrograms give us.

---

## Part 7: Spectrograms

A **spectrogram** solves the limitation of a single FFT by computing
many short FFTs on overlapping windows of the signal, then displaying
the results as an image:

- **x-axis:** time
- **y-axis:** frequency
- **Color:** amplitude (energy) — brighter/warmer = more energy at that
  frequency at that time

This is called a **Short-Time Fourier Transform (STFT)** — instead of
one FFT over the whole signal, we slide a window across the signal and
compute the FFT at each position.

In [ ]:
# Compute the Short-Time Fourier Transform
stft_result = librosa.stft(speech_signal)

# Convert amplitude to decibels (log scale) for better visualization
spectrogram_db = librosa.amplitude_to_db(np.abs(stft_result), ref=np.max)

plt.figure(figsize=(12, 5))
librosa.display.specshow(
    spectrogram_db,
    sr=speech_sr,
    x_axis='time',
    y_axis='hz',
)
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram of kdt_001.wav')
plt.tight_layout()
plt.show()

**Reading a spectrogram:**

- Time flows left to right (just like a waveform)
- Low frequencies are at the bottom, high frequencies at the top
- Bright/warm bands = energy concentrations at those frequencies
- Dark regions = little energy (quiet, or no sound at that frequency)

Compare the spectrogram to the waveform above. The parts with large
amplitude in the waveform should correspond to bright regions in the
spectrogram. Silent gaps appear as dark vertical bands.

Now that we can read spectrograms, let's connect what we see to what we
know about speech sounds.

### Fundamental frequency (F0)

When you produce a **voiced** sound (like a vowel), your vocal folds
vibrate at a regular rate. This rate is the **fundamental frequency (F0)**.

- Low male voice: F0 ~ 85–180 Hz
- Female voice: F0 ~ 165–255 Hz
- Child's voice: F0 ~ 250–400 Hz

F0 is what we perceive as **pitch**.

### Harmonics

Vocal fold vibration doesn't just produce energy at F0 — it also
produces energy at integer multiples of F0, called **harmonics**. If
F0 = 100 Hz, the harmonics are at 200, 300, 400, 500 Hz, and so on.

On a spectrogram, harmonics appear as **evenly spaced horizontal bands**
during voiced sounds.

### Voiced vs. voiceless sounds

| Sound type | Examples | Spectrogram signature |
|-----------|----------|----------------------|
| **Voiced** (vowels, nasals, /z/, /v/) | [a], [i], [m], [z] | Clear harmonic bands (evenly spaced lines) |
| **Voiceless fricatives** | [s], [f], [ʃ] | Noisy energy ("fuzzy" with no clear bands), concentrated in higher frequencies |
| **Stops** (closure phase) | [p], [t], [k] | Brief silence (gap) followed by a burst of noise |

The key spectral difference is whether a sound has **harmonic structure**
(discrete peaks at regular intervals) or looks like **noise** (energy spread
across frequencies with no peaks). We can see this directly by comparing the
dulcimer note from Part 6 with white noise:

In [ ]:
# Load white noise (16 kHz)
noise_signal, noise_sr = librosa.load('../audio/white_noise_16k.wav', sr=None)

noise_fft = np.fft.fft(noise_signal)
noise_freqs = np.fft.fftfreq(len(noise_signal), d=1/noise_sr)

fig, axes = plt.subplots(2, 1, figsize=(12, 5))

# Top: dulcimer — harmonic peaks (dulcimer_signal loaded in Part 6)
pos_d = dulcimer_freqs_pos <= 3000
axes[0].plot(dulcimer_freqs_pos[pos_d], dulcimer_magnitude[pos_d])
axes[0].set_title('Harmonic spectrum: dulcimer C note (like a vowel)')
axes[0].set_ylabel('Magnitude')

# Bottom: white noise — energy spread flat across all frequencies
pos_n = noise_freqs >= 0
axes[1].plot(noise_freqs[pos_n], np.abs(noise_fft[pos_n]))
axes[1].set_xlim(0, 8000)
axes[1].set_title('Noise spectrum: white noise (like a fricative)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')

plt.tight_layout()
plt.show()

Voiced sounds (like vowels) show a harmonic spectrum — discrete peaks like
the dulcimer above. Voiceless fricatives (/s/, /ʃ/, /f/) show a noise-like
spectrum — energy spread across a range of frequencies with no clear peaks.

Now let's look for these patterns in the spectrogram of our speech
recording:

In [ ]:
# Plot the full spectrogram with waveform above for reference
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Waveform on top
librosa.display.waveshow(speech_signal, sr=speech_sr, ax=axes[0])
axes[0].set_title('Waveform')
axes[0].set_xlabel('')

# Spectrogram below
librosa.display.specshow(
    spectrogram_db,
    sr=speech_sr,
    x_axis='time',
    y_axis='hz',
    ax=axes[1],
)
axes[1].set_title('Spectrogram')

plt.tight_layout()
plt.show()

Look for these features in the spectrogram:

- **Vowels**: Bright horizontal bands (harmonics) with darker gaps between
  them. The spacing between bands equals F0.
- **Fricatives**: A "fuzzy" cloud of noise energy, often in higher
  frequencies. /s/ and /ʃ/ are especially visible as high-frequency noise.
- **Silences/stops**: Dark vertical bands where there's little energy at
  any frequency.

**Preview for next time:** You might also notice that different vowels
have their energy concentrated at different frequencies. These energy
concentrations are called **formants**, and they're the key to
understanding vowel identity. We'll measure formants next class using
Praat.

---

## Part 9: Standard Speech Datasets

Just as NLP has standard text corpora (Brown, Penn Treebank), speech
research has standard datasets. Here are some of the most widely used:

| Dataset | Size | Contents | Access |
|---------|------|----------|--------|
| **TIMIT** | ~5 hrs | 630 speakers reading phonetically balanced sentences; hand-aligned phoneme boundaries | Paid (LDC) |
| **LibriSpeech** | ~1000 hrs | Read English audiobook speech; clean and "other" subsets | Free |
| **Common Voice** (Mozilla) | 20,000+ hrs | Crowdsourced read speech in 100+ languages | Free |
| **VoxForge** | Varies | Crowdsourced speech for open-source speech recognition | Free |

**TIMIT** is the classic: small but very carefully annotated (every
phoneme boundary is marked by hand). It's been the benchmark for phonetic
research for decades, though it's showing its age (recorded in 1986,
limited speaker diversity).

**LibriSpeech** and **Common Voice** are much larger and freely
available, making them popular for speech recognition and increasingly
for linguistic research too.

If you're thinking about a term project involving speech data, Common
Voice is a good starting point — it's free, multilingual, and large
enough to do real analysis.

---

## Looking Ahead

Today we covered the fundamentals of digital audio: how sound works
physically, how it's captured digitally, and how to visualize it as
waveforms, spectra, and spectrograms.

Next time (Speech Data II), we'll get hands-on with:
- **Praat** — the standard tool for phonetic analysis
- **Formants** — resonances of the vocal tract that distinguish vowels
- **The F1/F2 vowel space** — how formant measurements map to articulatory
  descriptions (height, backness)
- **Annotation** — TextGrids, ELAN, and forced alignment for time-aligning
  transcriptions to audio

---

## Exercises

Try these on your own!

### Exercise 1: FFT of a sine wave file

Load `sine_200Hz.wav` from the `audio/` directory. Compute its FFT and
plot the magnitude spectrum (positive frequencies only, up to 500 Hz).
Verify that the peak is at 200 Hz.

**Hint:** Follow the pattern from Part 6 — use `np.fft.fft()` and
`np.fft.fftfreq()`, then filter to positive frequencies.

In [ ]:
# Your code here


### Exercise 2: Build your own composite signal

Create a composite signal from 4 or more sine waves of your choosing
(pick different frequencies and amplitudes). Plot both the waveform (50 ms
window) and the FFT magnitude spectrum. Verify that the FFT peaks match
the frequencies you used.

**Hint:** Use `np.sin(2 * np.pi * freq * t)` for each component, then
add them together.

In [ ]:
# Your code here


### Exercise 3: Reading a spectrogram

Load `kdt_001.wav` and plot its spectrogram (you can copy the code from
Part 7). Identify at least two visually distinct regions in the
spectrogram and describe what you see. For example:

- "From ~0.3 to ~0.5 seconds there are clear horizontal bands — this
  looks like a vowel."
- "Around ~0.8 seconds there's high-frequency noise — this might be a
  fricative like /s/."

Write your observations as a markdown cell below your spectrogram plot.

In [ ]:
# Your code here


---

## Quick Reference

### librosa

| Code | What it does |
|------|-------------|
| `librosa.load(path, sr=None)` | Load audio file; returns (signal, sample_rate) |
| `librosa.display.waveshow(y, sr)` | Plot a waveform |
| `librosa.stft(y)` | Compute the Short-Time Fourier Transform |
| `librosa.amplitude_to_db(S)` | Convert amplitude spectrogram to decibels |
| `librosa.display.specshow(S, sr, ...)` | Display a spectrogram as an image |

### numpy (signal processing)

| Code | What it does |
|------|-------------|
| `np.sin(2 * np.pi * freq * t)` | Generate a sine wave |
| `np.fft.fft(signal)` | Compute the Fast Fourier Transform |
| `np.fft.fftfreq(n, d=1/sr)` | Get frequency bins for FFT output |
| `np.abs(fft_result)` | Get magnitude (amplitude) from FFT |

### Key terms

| Term | Definition |
|------|------------|
| **Sample rate** | Number of amplitude measurements per second (Hz) |
| **Bit depth** | Precision of each sample (e.g., 16-bit = 65,536 levels) |
| **Nyquist frequency** | sample_rate / 2; highest representable frequency |
| **Aliasing** | Distortion when recording frequencies above Nyquist |
| **FFT** | Algorithm that decomposes a signal into frequency components |
| **Spectrum** | Frequency content of a signal (output of FFT) |
| **Spectrogram** | Time × frequency × amplitude visualization (many short FFTs) |
| **STFT** | Short-Time Fourier Transform; the method behind spectrograms |
| **F0** | Fundamental frequency; rate of vocal fold vibration |
| **Harmonics** | Integer multiples of F0 |
| **Formants** | Resonant frequencies of the vocal tract (next lecture) |